# Hallmark — train the visual forgery model on Kaggle

1. **Add data** → search *certificate-forgery-dataset* (harikrish03) and attach it.
2. Make the Hallmark code available: attach this repository as a Kaggle dataset named `hallmark-code` (upload the repo zip), **or** set `REPO_URL` below to your GitHub repository.
3. **Settings → Accelerator → GPU T4**, then *Run all*.
4. Download `hallmark_model_artifacts.zip` from the output panel and unzip it into `models/artifacts/`.

In [ ]:
import os, glob, subprocess
DATA = '/kaggle/input/certificate-forgery-dataset'
print(os.path.exists(DATA))
!find $DATA -maxdepth 3 -type d | head -40
!find $DATA -type f | wc -l

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<your-user>/hallmark.git'
if REPO_URL:
    !git clone -q $REPO_URL /kaggle/working/hallmark
else:
    src = glob.glob('/kaggle/input/hallmark-code*/**/models/train.py', recursive=True)
    assert src, 'Attach the Hallmark code as a dataset or set REPO_URL'
    root = os.path.dirname(os.path.dirname(src[0]))
    !cp -r $root /kaggle/working/hallmark
%cd /kaggle/working/hallmark

## 1. Inspect the dataset
Check the class mapping, duplicates and the grouping strategy before training. Class folders, label CSVs and YOLO/Roboflow exports are detected automatically. If both classes are not found, pass `--label-map 'folderA=authentic,folderB=forged'`.

In [ ]:
!python models/train.py --data-dir $DATA --out /kaggle/working/artifacts --inspect-only
import json; print(json.dumps(json.load(open('/kaggle/working/artifacts/data_report.json')), indent=1))

## 2. Train, calibrate and evaluate

In [ ]:
# method 'auto' picks linear_probe for few source documents (this dataset: 15) and finetune otherwise
!pip install -q efficientnet_pytorch==0.7.1
!python models/train.py --data-dir $DATA --out /kaggle/working/artifacts

In [ ]:
cfg = json.load(open('/kaggle/working/artifacts/model_config.json'))
print('validation gate:', json.dumps(cfg['validation'], indent=1))
print(json.dumps(cfg['metrics']['test_calibrated'], indent=1))
print('temperature', cfg['temperature'], 'sizes', cfg['split_sizes'])
from IPython.display import Image; Image('/kaggle/working/artifacts/reliability_test.png')

## 3. Package the artifacts

In [ ]:
!cd /kaggle/working && zip -qr hallmark_model_artifacts.zip artifacts && ls -lh hallmark_model_artifacts.zip